In [449]:

import os 
import datetime


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm  
from scipy import stats
from scipy.optimize import minimize
from sklearn.metrics import log_loss


### Initial settings

In [450]:
optimize_params = True

### Load Data

In [436]:


def load_projections():
    
    projections_files = os.listdir('../data/picks_v2/')
    projections_files = [f for f in projections_files if f.endswith('.csv')]
    projection_paths = [os.path.join('../data/picks_v2/', f) for f in projections_files]
    projections = pd.concat([pd.read_csv(f) for f in projection_paths], ignore_index=True)
    projections = projections.drop(columns=['ML' ,'Fudge'])
    projections['Track'] = projections['Track'].map({
        'Churchill':'churchill-downs',
        'Keeneland':'keeneland',
        'Pimlico':'pimlico'
    })
    projections.columns=['date','track','race','horse','form','time','composite','has_cappers','sweep','hersh','consensus','peck','drf_sum']
    projections['date'] = pd.to_datetime(projections['date']).apply(lambda x: x.strftime('%Y-%m-%d'))

    return projections


def load_results():
    
    results_files = os.listdir('../data/entries/')
    results_files = [f for f in results_files if f.endswith('.csv')]
    results_files = [f for f in results_files if '2025' in f]
    results_paths = [os.path.join('../data/entries/', f) for f in results_files]
    results = pd.concat([pd.read_csv(f) for f in results_paths], ignore_index=True)
    
    return results

projections = load_projections()
results = load_results()

backtest_data = pd.merge(projections, results, on=['date','track','race','horse'], how='left')



### Process

In [439]:

def add_cols(df):
    
    df['won'] = np.where(df['finish'] == 1, 1, 0)
    df['date'] = pd.to_datetime(df['date']) 
    todays_date = datetime.datetime.now()
    df = df.loc[df['date'] < todays_date].copy().reset_index(drop=True)
    df['global_race_number'] = df.groupby(['date','track','race']).ngroup() + 1
    df = df.loc[df['running'] == 1].copy().reset_index(drop=True)
    
    df['race_3q'] = df.groupby(['global_race_number'])['time'].transform(lambda x: x.quantile(0.75))
    df['ahead_of_3q'] = df['time'] - df['race_3q']
    df['ahead_of_mean'] = df['form'] - df.groupby(['global_race_number'])['form'].transform('mean') 
    df['behind_max'] = df['form'] - df.groupby(['global_race_number'])['form'].transform('max') 
    df['num_horses'] = df.groupby(['global_race_number'])['composite'].transform('count') 
    df['num_horses'] = df['num_horses'].astype(int)
    # data['num_horses_sq'] = data['num_horses'] ** 2
    df['place'] = np.where(df['finish'] <= 2, 1, 0)
    df['show'] = np.where(df['finish'] <= 3, 1, 0)
    df['num_horses_sq'] = df['num_horses'] ** 2
    
    return df

backtest_data = add_cols(backtest_data)





In [443]:

def calculate_win_probabilities_df_vectorized(df, composite_col='composite', std_dev=4, 
                                             degrees_of_freedom=5, num_simulations=100000):
    """
    Vectorized implementation for calculating winning probabilities.
    """
    # Create a copy of the input DataFrame to avoid modifying the original
    result_df = df.copy()
    
    # Initialize a win_probability column with zeros
    result_df['t_sim_prob'] = 0.0
    
    # Create t-distribution with specified degrees of freedom
    t_dist = stats.t(df=degrees_of_freedom)
    
    # Group by date, track, and race
    for (date, track, race), race_df in result_df.groupby(['date', 'track', 'race']):
        # Get indices for this race
        race_indices = race_df.index
        num_horses = len(race_indices)
        
        # Skip races with only one horse
        if num_horses <= 1:
            continue
            
        # Extract horse composite values for this race
        composite_values = race_df[composite_col].values
        
        # Reshape for broadcasting
        composite_values = composite_values.reshape(num_horses, 1)
        
        # Generate all random effects at once (num_horses x num_simulations)
        random_effects = t_dist.rvs(size=(num_horses, num_simulations)) * std_dev
        
        # Calculate all finish positions at once
        finishes = composite_values + random_effects
        
        # Find winner for each simulation
        winners = np.argmax(finishes, axis=0)
        
        # Count wins for each horse
        win_counts = np.bincount(winners, minlength=num_horses)
        win_probabilities = win_counts / num_simulations
        
        # Assign back to DataFrame
        result_df.loc[race_indices, 't_sim_prob'] = win_probabilities
    
    return result_df


In [434]:
## use thisinstead of the above

def objective_function(params, data=None):
    """
    Objective function to minimize. Calculates log loss with the given parameters.
    
    Args:
        params: array of parameters [std_dev, df]
        data: DataFrame containing the necessary data
    
    Returns:
        log loss value to minimize
    """
    # Extract parameters from the array
    std_dev = params[0]
    degrees_of_freedom = params[1]
    
    result_df = calculate_win_probabilities_df_vectorized(
        data, 
        composite_col='composite', 
        std_dev=std_dev,  # Use the first parameter as std_dev
        degrees_of_freedom=degrees_of_freedom,  # Use the second parameter as df
        num_simulations=10000  # Use more simulations for final model
    )
    
    return log_loss(result_df['won'], result_df['projected_win_probability'])

def optimize_parameters(data, initial_std_dev=4.0, initial_df=1):
    """
    Optimize the std_dev and degrees_of_freedom parameters to minimize log loss.
    
    Args:
        data: DataFrame containing the necessary data
        initial_std_dev: Initial value for standard deviation
        initial_df: Initial value for degrees of freedom
    
    Returns:
        Tuple of optimal (std_dev, degrees_of_freedom)
    """
    # Create initial parameter array
    initial_params = np.array([initial_std_dev, initial_df])
    
    # Define bounds for parameters
    bounds = [(1, 10), (0.1, 2)]  # (std_dev, degrees_of_freedom)
    
    # Use Nelder-Mead optimization (doesn't require gradients)
    result = minimize(
        objective_function,
        initial_params,
        args=(data,),
        method='Nelder-Mead',
        bounds=bounds,
        options={'maxiter': 250, 'disp': True}
    )
    
    
    # Return the optimal parameters
    return result.x[0], result.x[1]  # std_dev, degrees_of_freedom

## run on 5/22
if optimize_params:
    optimal_std_dev, optimal_df = optimize_parameters(backtest_data, initial_std_dev=4.0, initial_df=1)
    print(f"Optimal parameters: std_dev={optimal_std_dev:.4f}, degrees_of_freedom={optimal_df:.4f}")

# Optimal parameters: std_dev=4.6500, degrees_of_freedom=0.8875

Optimal parameters: std_dev=4.6820, degrees_of_freedom=0.8492


C:\Users\Blake\AppData\Local\Temp\ipykernel_43736\2624501885.py:47: RuntimeWarning: Maximum number of iterations has been exceeded.
  result = minimize(


In [444]:

if not optimize_params:
    optimal_std_dev = 4.6500
    optimal_df = 0.8875

# Use optimized parameters for final model
data = calculate_win_probabilities_df_vectorized(
    backtest_data, 
    composite_col='composite', 
    std_dev=optimal_std_dev, 
    degrees_of_freedom=optimal_df, 
    num_simulations=100000     # Use more simulations for final model
)

print(f"Optimal parameters: std_dev={optimal_std_dev:.4f}, degrees_of_freedom={optimal_df:.4f}")


Optimal parameters: std_dev=4.6820, degrees_of_freedom=0.8492


In [447]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict 
from sklearn.metrics import log_loss

X = data[['ahead_of_3q', 'ahead_of_mean','behind_max', 'num_horses', 'num_horses_sq']] 
y = data['won']
win_model = LogisticRegression()
y_pred = cross_val_predict(win_model, X, y, cv=10, method='predict_proba')[:, 1]
log_loss_score = log_loss(y, y_pred)
print(f"Log Loss Score: {log_loss_score:.4f}")

## see coefficients 
win_model.fit(X, y)
coefficients = win_model.coef_[0]
intercept = win_model.intercept_[0]
data['raw_win_prob'] = y_pred
# data = apply_optimal_temperature(data, optimal_temp=1, race_col='global_race_number', raw_prob_col='raw_win_prob', new_col='calibrated_prob')
data['regr_prob'] = data['raw_win_prob']/ data.groupby(['global_race_number'])['raw_win_prob'].transform('sum') 
log_loss(data['won'], data['t_sim_prob'])
coefficients, intercept


Log Loss Score: 0.3211


(array([ 0.02113763,  0.10884669,  0.06202547, -0.21942013,  0.0047018 ]),
 -0.2928831643094048)

In [448]:

data['pred_final'] = data[['t_sim_prob\
    ','regr_prob']].mean(axis=1)
## normalize
data['pred_final'] = data['pred_final'] / data.groupby(['global_race_number'])['pred_final'].transform('sum')


KeyError: "['t_sim_prob    '] not in index"